In [1]:
import json
from collections import Counter
import statistics

path = "../../results/phase1_easy/planner_coder/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)

total = len(data)
passed = sum(1 for x in data if x["status"] == "pass")
timeout = sum(1 for x in data if x["status"] == "timeout")
failed = total - passed - timeout

print("=" * 60)
print(f"📂 File: {path}")
print("📊 Overall")
print(f"Total: {total}")
print(f"Pass: {passed}")
print(f"Fail: {failed}")
print(f"Timeout: {timeout}")
print(f"Pass@1: {passed / total:.4f}" if total > 0 else "Pass@1: N/A")

📂 File: ../../results/phase1_easy/planner_coder/details.jsonl
📊 Overall
Total: 164
Pass: 82
Fail: 82
Timeout: 0
Pass@1: 0.5000


In [2]:
error_counter = Counter(
    x["error_type"] for x in data if x["error_type"] is not None
)

print("\n📉 Error Type Distribution")
if error_counter:
    for k, v in error_counter.most_common():
        print(f"{k}: {v}")
else:
    print("(none)")


📉 Error Type Distribution
name_error: 49
syntax_error: 19
assertion_error: 14


In [3]:
total_latencies = [x["latency_sec"] for x in data if x.get("latency_sec") is not None]
planner_latencies = [x["planner_latency_sec"] for x in data if x.get("planner_latency_sec") is not None]
coder_latencies = [x["coder_latency_sec"] for x in data if x.get("coder_latency_sec") is not None]

if total_latencies:
    print("\n⏱ Total Latency")
    print(f"Avg: {statistics.mean(total_latencies):.3f}s")
    print(f"Max: {max(total_latencies):.3f}s")

if planner_latencies:
    print("\n🧠 Planner Latency")
    print(f"Avg: {statistics.mean(planner_latencies):.3f}s")
    print(f"Max: {max(planner_latencies):.3f}s")

if coder_latencies:
    print("\n💻 Coder Latency")
    print(f"Avg: {statistics.mean(coder_latencies):.3f}s")
    print(f"Max: {max(coder_latencies):.3f}s")


⏱ Total Latency
Avg: 3.714s
Max: 6.598s

🧠 Planner Latency
Avg: 2.086s
Max: 3.295s

💻 Coder Latency
Avg: 1.628s
Max: 3.312s


In [4]:
planner_lengths = [len(x.get("planner_output", "")) for x in data]
coder_lengths = [len(x.get("raw_output", "")) for x in data]

print("📝 Planner Output Length")
print(f"Avg chars: {statistics.mean(planner_lengths):.1f}")
print(f"Max chars: {max(planner_lengths)}")

print("\n📦 Raw Coder Output Length")
print(f"Avg chars: {statistics.mean(coder_lengths):.1f}")
print(f"Max chars: {max(coder_lengths)}")

📝 Planner Output Length
Avg chars: 721.1
Max chars: 1302

📦 Raw Coder Output Length
Avg chars: 380.1
Max chars: 1322


In [5]:
planner_lengths = [len(x.get("planner_output", "")) for x in data]
coder_lengths = [len(x.get("raw_output", "")) for x in data]

print("📝 Planner Output Length")
print(f"Avg chars: {statistics.mean(planner_lengths):.1f}")
print(f"Max chars: {max(planner_lengths)}")

print("\n📦 Raw Coder Output Length")
print(f"Avg chars: {statistics.mean(coder_lengths):.1f}")
print(f"Max chars: {max(coder_lengths)}")

📝 Planner Output Length
Avg chars: 721.1
Max chars: 1302

📦 Raw Coder Output Length
Avg chars: 380.1
Max chars: 1322


In [6]:
print("\n❌ Sample Failures (up to 3)")
failures = [x for x in data if x["status"] == "fail"][:3]

if not failures:
    print("No failures found.")
else:
    for f in failures:
        print("-" * 40)
        print(f"Task: {f['task_id']}")
        print(f"Error Type: {f['error_type']}")
        print(f"Error Msg: {f['error_message']}")


❌ Sample Failures (up to 3)
----------------------------------------
Task: HumanEval/1
Error Type: assertion_error
Error Msg: Traceback (most recent call last):
  File "/tmp/tmpxvntsvde.py", line 43, in <module>
    check(separate_paren_groups)
  File "/tmp/tmpxvntsvde.py", line 41, in check
    assert candidate('( ) (( )) (( )( ))') == ['()', '(())', '(()())']
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError

----------------------------------------
Task: HumanEval/2
Error Type: name_error
Error Msg: Traceback (most recent call last):
  File "/tmp/tmp_yc4yqsu.py", line 12, in <module>
    s
NameError: name 's' is not defined

----------------------------------------
Task: HumanEval/7
Error Type: syntax_error
Error Msg:   File "/tmp/tmp5pxquz2w.py", line 38
    if
      ^
SyntaxError: invalid syntax



In [7]:
samples = data[:2]

for s in samples:
    print("=" * 100)
    print(f"TASK: {s['task_id']}")
    print("\n[PLANNER OUTPUT]")
    print(s.get("planner_output", "")[:1000])
    print("\n[RAW CODER OUTPUT]")
    print(s.get("raw_output", "")[:1000])
    print("\n[GENERATED CODE]")
    print(s.get("generated_code", "")[:1000])
    print(f"\n[STATUS] {s['status']} / {s['error_type']}")

TASK: HumanEval/0

[PLANNER OUTPUT]
: Sort the input list of numbers.
2: Iterate through the sorted list and compare adjacent elements.
3: If the difference between any two adjacent elements is less than the threshold, return True.
4: If no such pair is found after checking all adjacent pairs, return False.
Edge Cases/Constraints:
- The list can be empty, in which case the function should return False.
- The threshold must be non-negative, as negative thresholds do not make sense in this context.
- The function should handle floating-point precision issues gracefully, possibly by using a small epsilon value when comparing differences.

[RAW CODER OUTPUT]
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    if threshold < 0:
        raise ValueError("Threshold must be non-negative")
    
    epsilon = 1e-9
    
    # Sort the list of numbers
    numbers.sort()
    
    # Iterate through the sorted list and compare adjacent eleme

## planner heuristic 체크

In [8]:
added_assumption_count = 0
code_leak_count = 0

for row in data:
    planner_output = row.get("planner_output", "").lower()

    if "def " in planner_output or "return " in planner_output:
        code_leak_count += 1

    if (
        "must be non-negative" in planner_output
        or "raise valueerror" in planner_output
        or "epsilon" in planner_output
    ):
        added_assumption_count += 1

print("🔍 Planner Output Heuristics")
print(f"Planner outputs containing code-like text: {code_leak_count}")
print(f"Planner outputs with possible extra assumptions: {added_assumption_count}")

🔍 Planner Output Heuristics
Planner outputs containing code-like text: 141
Planner outputs with possible extra assumptions: 2


In [9]:
# 특정 실패 케이스
failures = [x for x in data if x["status"] == "fail"]

case = failures[0]
print("=" * 100)
print(f"TASK: {case['task_id']}")
print(f"STATUS: {case['status']}")
print(f"ERROR TYPE: {case['error_type']}")
print("\n[PLANNER OUTPUT]")
print(case["planner_output"])
print("\n[RAW CODER OUTPUT]")
print(case["raw_output"])
print("\n[GENERATED CODE]")
print(case["generated_code"])
print("\n[ERROR MESSAGE]")
print(case["error_message"])

TASK: HumanEval/1
STATUS: fail
ERROR TYPE: assertion_error

[PLANNER OUTPUT]
 - Initialize an empty list to store the result.
2 - Iterate through the characters in the input string, ignoring spaces.
3 - Use a counter to keep track of the balance between opening and closing parentheses.
4 - When encountering an opening parenthesis, increment the counter.
5 - When encountering a closing parenthesis, decrement the counter.
6 - Whenever the counter returns to zero, it indicates that a complete, balanced group has been found.
7 - Extract this group from the current position back to where the last counter reset was, add it to the result list.
8 - Continue until all characters have been processed.
9 - Return the result list.

Edge Cases/Constraints:
- The input string may contain spaces which should be ignored.
- The input string may be empty, in which case an empty list should be returned.
- The input string may contain unbalanced parentheses, but only fully balanced groups should be returne

In [10]:
import pandas as pd

df = pd.DataFrame(data)
df.head()

,dataset,task_id,entry_point,method,planner_model_name,coder_model_name,status,passed,timeout,planner_prompt,planner_output,coder_prompt,raw_output,generated_code,error_type,error_message,planner_latency_sec,coder_latency_sec,latency_sec
0,humaneval,HumanEval/0,has_close_elements,planner_coder,Qwen/Qwen2.5-Coder-7B-Instruct,Qwen/Qwen2.5-Coder-7B-Instruct,pass,True,False,You are a planning assistant for Python progra...,: Sort the input list of numbers.\n2: Iterate ...,You are a Python coding assistant.\n\nWrite a ...,```python\nfrom typing import List\n\n\ndef ha...,from typing import List\n\ndef has_close_eleme...,None,None,1.744568,1.523317,3.267885
1,humaneval,HumanEval/1,separate_paren_groups,planner_coder,Qwen/Qwen2.5-Coder-7B-Instruct,Qwen/Qwen2.5-Coder-7B-Instruct,fail,False,False,You are a planning assistant for Python progra...,- Initialize an empty list to store the resul...,You are a Python coding assistant.\n\nWrite a ...,```python\nfrom typing import *\nfrom collecti...,from typing import *\nfrom collections import ...,assertion_error,"Traceback (most recent call last):\n File ""/t...",2.482558,1.589465,4.072024
2,humaneval,HumanEval/2,truncate_number,planner_coder,Qwen/Qwen2.5-Coder-7B-Instruct,Qwen/Qwen2.5-Coder-7B-Instruct,fail,False,False,You are a planning assistant for Python progra...,\n\n,You are a Python coding assistant.\n\nWrite a ...,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,\n\ndef truncate_number(number: float) -> floa...,name_error,"Traceback (most recent call last):\n File ""/t...",0.110438,3.271655,3.382093
3,humaneval,HumanEval/3,below_zero,planner_coder,Qwen/Qwen2.5-Coder-7B-Instruct,Qwen/Qwen2.5-Coder-7B-Instruct,pass,True,False,You are a planning assistant for Python progra...,:\n. Initialize a variable `balance` to 0 to k...,You are a Python coding assistant.\n\nWrite a ...,```python\nfrom typing import List\n\n\ndef be...,from typing import List\n\ndef below_zero(oper...,None,None,1.972695,0.709328,2.682023
4,humaneval,HumanEval/4,mean_absolute_deviation,planner_coder,Qwen/Qwen2.5-Coder-7B-Instruct,Qwen/Qwen2.5-Coder-7B-Instruct,pass,True,False,You are a planning assistant for Python progra...,:\n1. Calculate the mean of the input list.\n2...,You are a Python coding assistant.\n\nWrite a ...,```python\nfrom typing import List\n\n\ndef me...,from typing import List\n\ndef mean_absolute_d...,None,None,1.218899,1.089001,2.307900


In [11]:
df["status"].value_counts()

status
pass    82
fail    82
Name: count, dtype: int64

In [13]:
df["error_type"].value_counts(dropna=False)

error_type
None               82
name_error         49
syntax_error       19
assertion_error    14
Name: count, dtype: int64

In [14]:
df[["task_id", "status", "planner_latency_sec", "coder_latency_sec", "latency_sec"]].head(10)

,task_id,status,planner_latency_sec,coder_latency_sec,latency_sec
0,HumanEval/0,pass,1.744568,1.523317,3.267885
1,HumanEval/1,fail,2.482558,1.589465,4.072024
2,HumanEval/2,fail,0.110438,3.271655,3.382093
3,HumanEval/3,pass,1.972695,0.709328,2.682023
4,HumanEval/4,pass,1.218899,1.089001,2.307900
5,HumanEval/5,pass,3.239675,2.031337,5.271012
6,HumanEval/6,pass,2.331770,1.437585,3.769355
7,HumanEval/7,fail,1.908441,3.286851,5.195293
8,HumanEval/8,pass,1.207467,0.987353,2.194820
9,HumanEval/9,pass,1.576492,0.987646,2.564138


In [15]:
# planner output이 과한 케이스
df[df["planner_output"].str.contains("must be non-negative|epsilon|raise ValueError", case=False, na=False)][
    ["task_id", "planner_output"]
].head(10)

,task_id,planner_output
0,HumanEval/0,: Sort the input list of numbers.\n2: Iterate ...
139,HumanEval/139,: Implement the `special_factorial` function t...


In [16]:
# planner가 code leakage
df[df["planner_output"].str.contains(r"def |return ", case=False, na=False, regex=True)][
    ["task_id", "planner_output"]
].head(10)

,task_id,planner_output
0,HumanEval/0,: Sort the input list of numbers.\n2: Iterate ...
1,HumanEval/1,- Initialize an empty list to store the resul...
3,HumanEval/3,:\n. Initialize a variable `balance` to 0 to k...
4,HumanEval/4,:\n1. Calculate the mean of the input list.\n2...
5,HumanEval/5,": Intended Algorithm\nTo solve this problem, w..."
6,HumanEval/6,: \n1. Split the input string into individual ...
7,HumanEval/7,: Implement a function `filter_by_substring` t...
8,HumanEval/8,: Initialize sum as 0 and product as 1.\n2: It...
9,HumanEval/9,: \n1. Initialize an empty list to store the r...
11,HumanEval/11,: Implement a function `string_xor` that takes...


### generated_code / raw_output 길이 확인

In [17]:
df["planner_len"] = df["planner_output"].fillna("").map(len)
df["raw_output_len"] = df["raw_output"].fillna("").map(len)
df["generated_code_len"] = df["generated_code"].fillna("").map(len)

df[["task_id", "status", "planner_len", "raw_output_len", "generated_code_len"]].head(10)

,task_id,status,planner_len,raw_output_len,generated_code_len
0,HumanEval/0,pass,606,499,481
1,HumanEval/1,fail,985,528,511
2,HumanEval/2,fail,2,225,526
3,HumanEval/3,pass,683,229,215
4,HumanEval/4,pass,489,329,315
5,HumanEval/5,pass,1184,689,663
6,HumanEval/6,pass,848,511,497
7,HumanEval/7,fail,745,1322,1300
8,HumanEval/8,pass,413,295,277
9,HumanEval/9,pass,565,334,320
